# Cell Type Neighborhoods v1

Cell-type neighborhoods of the Visium cohort (Spencer Watson; moved from `analysis/survival/Cell_Neighborhoods.ipynb`).

Produces:
- Cell-type neighborhoods (`celltype_neighborhood_annotations_k10.csv` and centroids, used by
  every cell-type neighborhood analysis of the paper); off by default (`RUN_CLUSTERING = False`
  reads the frozen files).
- **Fig 4b**: one section coloured by cell-type neighborhood (next to its ECM neighborhoods).
- **Supp Fig 4g** and the numbers **global NMI = 0.14, per-sample median = 0.24**: concordance of
  the cell-type and ECM partitions.
- **Supp Fig 4a**: marker dotplot of the cell-type neighborhoods, every cell type.
- **Fig 4a**: marker dotplot, top 5 cell types of the selected neighborhoods (see the note at the
  cell: the published panel differs from this cell's output).
- **Supp Fig 4b**: neighborhood composition of every sample.

Survival of the cell-type neighborhoods (Supp Fig 4c–d) is in `15_neighborhood_survival.ipynb`.

Changes from the original notebook:
- Loading cells (2–12) replaced by the package: `neighborhoods.load_spots` (spots and
  coordinates with the loader exclusions of `load_spatial_adata`; the expression matrix is not
  loaded, no cell here uses it), `visium.load_c2l` / `add_c2l_to_obs` (cell2location means),
  `annotation.add_celltype_key(efs_copy=True)` (the EFS key the notebook read). The clinical
  table (cell 4) was not used and is not loaded.
- Clustering: cell 36 (and the Hellinger representation it needs, first part of cell 14) run only
  with `RUN_CLUSTERING = True` and write to `OUT`; otherwise the frozen annotation and centroid
  files are read (`config.CT_ANNOTATIONS_CSV`).
- Removed the first strategy (Leiden clusters then kNN, superseded by cell 36): Leiden/UMAP part
  of cell 14, cells 16–25 (cell 22 also wrote `celltype_neighborhood_annotations_k10.csv`, with
  labels not sorted by size), 29 (earlier version of cell 38), 31 (earlier version of cell 46),
  33 (cell-type composition bars), 34 (exploratory); cell 1 (versions → `output.versions`).
- Cells 38 and 40 no longer reload the ECM file (cell 27 does).
- Cells 42 and 44: the cell2location → curated-type block is `neighborhoods.c2l_curated_counts`
  and the composition / specificity / marker-score matrices are `neighborhoods.marker_scores`
  (same code).
- Figures go to `OUT` with `output.save` (paper panels also to the panels folder); cell 46's matrix
  is compared with the one of the last run instead of overwriting it in `Analysis_Repo/Notebooks`.
- Per-cell imports moved to the top.

In [ ]:
import gc

import anndata as ad
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
from matplotlib.colors import LinearSegmentedColormap, to_hex
from matplotlib.gridspec import GridSpec
from scipy.cluster.hierarchy import dendrogram, leaves_list, linkage
from scipy.sparse import eye as speye
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from gbmcollaboration.paper import annotation, config, neighborhoods, output, visium

In [ ]:
# Parameters
OUT = output.setup("12_Cell_Neighborhoods")
sc.set_figure_params(dpi=150, color_map="viridis_r")
sc.settings.verbosity = 0

# True: re-run the cell-type neighborhood k-means (writes the annotation and centroid files to OUT);
# False: read the frozen annotations that every analysis of the paper used
RUN_CLUSTERING = False
CT_ANNOTATIONS_PATH = config.CT_ANNOTATIONS_CSV
CT_CENTROIDS_PATH = CT_ANNOTATIONS_PATH.parent / "celltype_neighborhood_centroids_k10.csv"
ecm_csv_path = config.ECM_ANNOTATIONS_CSV

### Load Visium spots and Cell2Location abundances

Spots and coordinates with the exclusions of `load_spatial_adata` (no expression matrix: none of
these cells uses it), cell2location mean abundances, annotation key.

In [ ]:
obs, _ = neighborhoods.load_spots()
adata = ad.AnnData(obs=obs)
visium.add_c2l_to_obs(adata, visium.load_c2l())
annotation.add_celltype_key(adata, efs_copy=True)
print(adata)

### Cell-type neighborhoods: kNN neighborhoods from cell composition

Hellinger-transformed cell-type proportions per spot (first part of the original cell 14), averaged
over each spot and its grid neighbours, k-means with k = 10 (cell 36). Run once to write
`celltype_neighborhood_annotations_k10.csv`; with `RUN_CLUSTERING = False` the frozen file is read.

In [ ]:
if RUN_CLUSTERING:
    # ### Cluster Spots Based on Cell Type Abundance


    # User config
    EXCLUDE_CELLTYPES = []
    celltype_rep_key  = "X_celltype"

    # Build cell type abundance matrix from C2L obs columns
    c2l_prefix   = "meanscell_abundance_w_sf_"
    celltype_key = adata.uns["celltype_annotation_key"]

    c2l_cols    = [c for c in adata.obs.columns if c.startswith(c2l_prefix)]
    lvl4_in_key = [
        c.replace(c2l_prefix, "") for c in c2l_cols
        if c.replace(c2l_prefix, "") in celltype_key.index
        and c.replace(c2l_prefix, "") not in EXCLUDE_CELLTYPES
    ]
    c2l_cols_filt = [c2l_prefix + n for n in lvl4_in_key]

    print(f"Cell types: {len(c2l_cols)} total, {len(lvl4_in_key)} used "
          f"({len(c2l_cols) - len(lvl4_in_key)} excluded or unmatched)")

    X_ct = adata.obs[c2l_cols_filt].to_numpy(dtype=np.float32)

    # Diagnostics: how prevalent are NaNs?
    n_nan_cells = np.isnan(X_ct).sum()
    n_nan_rows  = np.isnan(X_ct).any(axis=1).sum()
    n_nan_cols  = np.isnan(X_ct).any(axis=0).sum()
    if n_nan_cells > 0:
        print(f"NaN diagnostic: {n_nan_cells:,} NaN cells across "
              f"{n_nan_rows:,} rows and {n_nan_cols} cell-type columns")

    # Replace NaNs with 0 (a NaN in one cell-type column for a spot just means
    # that cell type was not detected there; treating it as 0 abundance is correct)
    X_ct = np.nan_to_num(X_ct, nan=0.0)

    # Drop only spots whose entire abundance vector is zero (these carry no info)
    zero_rows = X_ct.sum(axis=1) == 0
    if zero_rows.any():
        print(f"Excluding {zero_rows.sum():,} spots with zero total abundance")

    valid_idx = np.where(~zero_rows)[0]
    if valid_idx.size == 0:
        raise RuntimeError(
            "No spots have nonzero cell-type abundance. Check the C2L merge step."
        )
    print(f"Valid spots: {valid_idx.size:,} of {adata.n_obs:,}")

    X_ct_valid = X_ct[valid_idx]

    # Convert counts to proportions per spot
    row_sums   = X_ct_valid.sum(axis=1, keepdims=True)
    X_prop     = X_ct_valid / row_sums

    # Hellinger transform: sqrt of proportions, suitable for compositional data,
    # stabilizes variance and makes Euclidean distance approximate Hellinger distance
    X_rep = np.sqrt(X_prop, dtype=np.float32)

    # Store representation back into adata for the valid spots only,
    # leaving NaN rows for excluded spots so downstream code can detect them
    X_full = np.full((adata.n_obs, X_rep.shape[1]), np.nan, dtype=np.float32)
    X_full[valid_idx] = X_rep
    adata.obsm[celltype_rep_key] = X_full
    adata.uns["celltype_rep_celltypes"] = lvl4_in_key

In [ ]:
if RUN_CLUSTERING:
    # ### Spatial Neighborhood Analysis from Cell Type Composition
    #
    # Build neighborhood vectors by averaging the Hellinger-transformed cell type
    # proportion vectors of each spot and its spatial neighbors. Cluster these
    # vectors with kmeans to define spatial niches that are directly interpretable
    # as average cell type compositions.



    # User config
    celltype_rep_key                = "X_celltype"
    nhood_key                       = "celltype_neighborhood"
    coord_type                      = "grid"
    n_rings                         = 1
    include_self                    = True
    kmeans_n_clusters               = 10
    kmeans_n_init                   = 25
    kmeans_random_state             = 42
    max_spots_per_sample_for_kmeans = None
    save_dir                        = OUT

    # Validate inputs
    req_cols = ["sample", "imagecol", "imagerow"]
    missing  = [c for c in req_cols if c not in adata.obs.columns]
    if missing:
        raise ValueError(f"Missing required obs columns: {missing}")
    if celltype_rep_key not in adata.obsm:
        raise ValueError(
            f"adata.obsm['{celltype_rep_key}'] not found. "
            f"Run the cell type clustering cell first to generate it."
        )

    X_rep = adata.obsm[celltype_rep_key]
    n_features = X_rep.shape[1]
    print(f"Cell type representation: {n_features} cell types, {X_rep.shape[0]:,} spots")

    # Per-sample spatial neighborhood averaging
    # For each spot, the neighborhood vector is the mean Hellinger-proportion
    # vector of its spatial neighbors (and itself, if include_self=True)
    samples       = adata.obs["sample"].unique().tolist()
    all_feats     = []
    all_barcodes  = []
    all_samples   = []

    for sid in samples:
        mask = (adata.obs["sample"] == sid).to_numpy()
        if not mask.any():
            continue

        ad_s = adata[mask].copy()

        # Drop spots with NaN in the representation (came from invalid C2L spots)
        valid = ~np.isnan(ad_s.obsm[celltype_rep_key]).any(axis=1)
        if valid.sum() < 10:
            continue
        ad_s = ad_s[valid].copy()

        if "spatial" not in ad_s.obsm:
            ad_s.obsm["spatial"] = ad_s.obs[["imagecol", "imagerow"]].to_numpy()

        # Build spatial neighbor graph (Visium hex grid topology)
        sq.gr.spatial_neighbors(ad_s, coord_type=coord_type, n_rings=n_rings)
        A = ad_s.obsp["spatial_connectivities"].tocsr().astype(np.float32)

        # Optionally include each spot itself in its own neighborhood
        if include_self:
            A = A + speye(A.shape[0], dtype=np.float32, format="csr")

        # Average the cell type representation across spatial neighbors
        X_self  = ad_s.obsm[celltype_rep_key].astype(np.float32)
        X_sum   = A.dot(X_self)
        n_neigh = np.asarray(A.sum(axis=1)).ravel()
        n_neigh[n_neigh == 0] = 1.0
        X_nb    = X_sum / n_neigh[:, None]

        all_feats.append(X_nb)
        all_barcodes.extend(ad_s.obs_names.tolist())
        all_samples.extend([sid] * ad_s.n_obs)

        del ad_s, A, X_self, X_sum, X_nb, n_neigh
        gc.collect()

    if not all_feats:
        raise RuntimeError("No neighborhood features generated; check inputs.")

    X_nbhd          = np.vstack(all_feats).astype(np.float32)
    all_barcodes    = np.array(all_barcodes)
    all_samples_arr = np.array(all_samples)
    del all_feats, all_samples
    gc.collect()

    print(f"Neighborhood feature matrix: {X_nbhd.shape}")

    # Sample-balanced subset for kmeans training (optional)
    rng = np.random.default_rng(kmeans_random_state)
    if max_spots_per_sample_for_kmeans is None:
        train_indices = np.arange(X_nbhd.shape[0], dtype=int)
    else:
        train_indices = []
        for sid in np.unique(all_samples_arr):
            idx_s = np.where(all_samples_arr == sid)[0]
            if idx_s.size > max_spots_per_sample_for_kmeans:
                idx_s = rng.choice(idx_s, size=max_spots_per_sample_for_kmeans, replace=False)
            train_indices.append(idx_s)
        train_indices = np.sort(np.concatenate(train_indices))

    print(f"kmeans training subset: {train_indices.size:,} spots "
          f"({train_indices.size / X_nbhd.shape[0]:.1%} of total)")

    # kmeans on neighborhood vectors directly (no PCA — interpretability matters
    # more than dimensionality here, and 72 features is not large for kmeans)
    km = KMeans(
        n_clusters=kmeans_n_clusters,
        n_init=kmeans_n_init,
        random_state=kmeans_random_state,
        verbose=0,
    )
    km.fit(X_nbhd[train_indices])
    labels_all = km.predict(X_nbhd).astype(np.int32)

    # Sort neighborhoods by size so neighborhood "0" is the most prevalent
    sizes_raw = pd.Series(labels_all).value_counts().sort_values(ascending=False)
    relabel   = {int(old): str(new) for new, old in enumerate(sizes_raw.index)}
    labels_str = pd.Series(labels_all).map(relabel).to_numpy()

    print(f"\nkmeans complete: {kmeans_n_clusters} neighborhoods")

    # Map labels back to full adata
    labels_ser           = pd.Series(labels_str, index=pd.Index(all_barcodes, name="barcode"))
    adata.obs[nhood_key] = labels_ser.reindex(adata.obs_names).astype("category")

    # Reporting
    sizes = adata.obs[nhood_key].value_counts(dropna=True).sort_index()
    patient_coverage = (
        adata.obs.dropna(subset=[nhood_key])
        .groupby(nhood_key, observed=True)["sample"]
        .nunique()
    )
    summary = pd.DataFrame({
        "n_spots": sizes,
        "n_patients": patient_coverage,
        "pct_spots": (sizes / sizes.sum() * 100).round(2),
    })
    print(f"\n{sizes.size} neighborhoods from {adata.obs['sample'].nunique()} samples:")
    print(summary.to_string())

    # Save the kmeans centroids (in Hellinger-proportion space) so the neighborhood
    # composition readout is directly available
    celltype_names = adata.uns.get("celltype_rep_celltypes", [f"ct_{i}" for i in range(n_features)])
    centroid_df = pd.DataFrame(
        km.cluster_centers_,
        columns=celltype_names,
    )
    # Reindex centroids to match the size-sorted label order
    inv_relabel = {v: k for k, v in relabel.items()}
    centroid_df = centroid_df.iloc[[inv_relabel[str(i)] for i in range(kmeans_n_clusters)]].reset_index(drop=True)
    centroid_df.index = [str(i) for i in range(kmeans_n_clusters)]
    centroid_df.index.name = nhood_key
    adata.uns[f"{nhood_key}_centroids_hellinger"] = centroid_df

    # Convert centroids back to proportion space (squared) for direct interpretation
    centroid_prop = centroid_df ** 2
    centroid_prop = centroid_prop.div(centroid_prop.sum(axis=1), axis=0)
    adata.uns[f"{nhood_key}_centroids_proportion"] = centroid_prop

    # Save neighborhood assignments to disk
    out_path = f"{save_dir}/celltype_neighborhood_annotations_k{kmeans_n_clusters}.csv"
    df_save  = adata.obs[[nhood_key, "sample"]].copy()
    df_save.index.name = "spot_id"
    df_save.to_csv(out_path)
    print(f"\nSaved {df_save.shape[0]:,} spots to {out_path}")

    # Save centroids (proportion space) for cross-session reference
    centroid_path = f"{save_dir}/celltype_neighborhood_centroids_k{kmeans_n_clusters}.csv"
    centroid_prop.to_csv(centroid_path)
    print(f"Saved neighborhood centroids (cell type proportions) to {centroid_path}")

In [ ]:
# Reload cell-type neighborhoods and their centroids
if not RUN_CLUSTERING:
    nhood_key = "celltype_neighborhood"
    ct_annot = pd.read_csv(CT_ANNOTATIONS_PATH, index_col="spot_id")
    adata.obs[nhood_key] = ct_annot[nhood_key].astype("Int64").astype("string").reindex(adata.obs_names).astype("category")
    centroid_prop = pd.read_csv(CT_CENTROIDS_PATH, index_col=0)
    centroid_prop.index = centroid_prop.index.astype(str)
    adata.uns[f"{nhood_key}_centroids_proportion"] = centroid_prop
    print(adata.obs[nhood_key].value_counts(dropna=False).sort_index().to_string())

### Load Saved ECM Neighborhood Annotations

In [ ]:
ecm_annot    = pd.read_csv(ecm_csv_path).set_index("spot_id")
adata.obs["ECM_neighborhood"] = (
    ecm_annot["ECM_neighborhood"]
    .reindex(adata.obs_names)
    .astype("category")
)

### Fig 4b: CT neighborhoods on sample

In [ ]:
# ### Plot Cell Type and ECM Neighborhoods Side-by-Side


# User config
sample_id    = "CH_G_196a_vis"
ct_nhood_key = "celltype_neighborhood"
ecm_nhood_key = "ECM_neighborhood"
spot_size    = 80
alpha        = 0.9
ct_cmap_name  = "tab20"
ecm_cmap_name = "tab20b"
unassigned_color = "#e8e8e8"

# Subset to the sample
ad = adata[adata.obs["sample"] == sample_id].copy()
if ad.n_obs == 0:
    raise ValueError(f"Sample '{sample_id}' not found in adata")

if "spatial" in ad.obsm:
    xy = ad.obsm["spatial"]
elif {"imagecol", "imagerow"}.issubset(ad.obs.columns):
    xy = ad.obs[["imagecol", "imagerow"]].to_numpy()
else:
    raise KeyError("Need obsm['spatial'] or obs['imagecol','imagerow'].")

# Build a label-to-dominant-celltype map for the cell type neighborhoods
# so the legend is biologically informative
centroid_key = f"{ct_nhood_key}_centroids_proportion"
nh_label_map = {}
if centroid_key in adata.uns:
    centroids = adata.uns[centroid_key]
    for nh in centroids.index:
        top_ct = centroids.loc[nh].sort_values(ascending=False).index[0]
        # Shorten very long cell type names for legend readability
        short = top_ct.replace("_", " ")
        if len(short) > 25:
            short = short[:22] + "…"
        nh_label_map[nh] = f"{nh}: {short}"
else:
    nh_label_map = {}

def make_color_map(series, cmap_name):
    cats = sorted(series.dropna().astype(str).unique(), key=lambda x: int(x))
    cmap = plt.get_cmap(cmap_name, max(len(cats), 1))
    return cats, {c: to_hex(cmap(i)) for i, c in enumerate(cats)}

# Use global category lists (across cohort), not just the current sample,
# so colors stay consistent between samples
def global_categories(key, cmap_name):
    if key not in adata.obs.columns:
        return [], {}
    series = adata.obs[key].astype("string").dropna()
    cats = sorted(series.unique(), key=lambda x: int(x))
    cmap = plt.get_cmap(cmap_name, max(len(cats), 1))
    return cats, {c: to_hex(cmap(i)) for i, c in enumerate(cats)}

ct_cats_all,  ct_cm  = global_categories(ct_nhood_key,  ct_cmap_name)
ecm_cats_all, ecm_cm = global_categories(ecm_nhood_key, ecm_cmap_name)

def color_array(series, color_map):
    s = series.astype("string")
    return s.map(lambda x: color_map.get(x, unassigned_color) if pd.notna(x) else unassigned_color).to_numpy()

ct_colors  = color_array(ad.obs[ct_nhood_key],  ct_cm)
ecm_colors = color_array(ad.obs[ecm_nhood_key], ecm_cm)

# Restrict legend to neighborhoods actually present in this sample
ct_present  = [c for c in ct_cats_all  if c in ad.obs[ct_nhood_key].astype("string").dropna().unique()]
ecm_present = [c for c in ecm_cats_all if c in ad.obs[ecm_nhood_key].astype("string").dropna().unique()]

n_ct_unassigned  = ad.obs[ct_nhood_key].isna().sum()
n_ecm_unassigned = ad.obs[ecm_nhood_key].isna().sum()

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

panels = [
    (axes[0], ct_colors,  ct_present,  ct_cm,  "Cell-type neighborhood",  n_ct_unassigned, nh_label_map),
    (axes[1], ecm_colors, ecm_present, ecm_cm, "ECM neighborhood",        n_ecm_unassigned, {}),
]

for ax, colors, present, cm, title, n_unassigned, label_map in panels:
    ax.scatter(xy[:, 0], xy[:, 1], c=colors, s=spot_size, edgecolors="none", alpha=alpha)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    panel_title = f"{sample_id}\n{title}"
    if n_unassigned > 0:
        panel_title += f"  ({n_unassigned:,} unassigned)"
    ax.set_title(panel_title, fontsize=12, fontweight="semibold", loc="left")

    handles = [mpatches.Patch(color=cm[k], label=label_map.get(k, k)) for k in present]
    if n_unassigned > 0:
        handles.append(mpatches.Patch(color=unassigned_color, label="unassigned"))
    ax.legend(
        handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0),
        frameon=False, fontsize=9, title="Niche", title_fontsize=10,
        handlelength=1.0, handleheight=1.0, labelspacing=0.4,
    )

plt.tight_layout()
output.save(OUT, f"{sample_id}_celltype_ECM_neighborhoods", fig,
            ad.obs[[ct_nhood_key, ecm_nhood_key, "imagecol", "imagerow"]], panel="Fig 4b")
plt.show()

### Supp Fig 4g: NMI comparison between CT and ECM neighborhoods

In [ ]:
# ### Concordance of Cell Type and ECM Neighborhoods
#
# Quantify how cell-type and ECM spatial niches overlap. NMI is computed
# both globally and per-sample so between-patient heterogeneity in the
# cell-ECM coupling is visible. The joint frequency matrix shows which
# niche pairs co-occur, with marginals showing each neighborhood's prevalence.


# User config
ct_nhood_key  = "celltype_neighborhood"
ecm_nhood_key = "ECM_neighborhood"
min_spots_per_sample = 100

# Spots with both assignments
df = adata.obs[[ct_nhood_key, ecm_nhood_key, "sample"]].dropna(
    subset=[ct_nhood_key, ecm_nhood_key]
).copy()
df[ct_nhood_key]  = df[ct_nhood_key].astype(str)
df[ecm_nhood_key] = df[ecm_nhood_key].astype(str)
print(f"Spots with both assignments: {len(df):,} from {df['sample'].nunique()} samples")

# Global metrics
nmi_global = normalized_mutual_info_score(df[ct_nhood_key], df[ecm_nhood_key])
ari_global = adjusted_rand_score(df[ct_nhood_key], df[ecm_nhood_key])
print(f"\nGlobal NMI: {nmi_global:.3f}   Global ARI: {ari_global:.3f}")
print("(NMI: 0 = independent, 1 = identical. "
      "ARI: 0 = chance, 1 = identical, can be negative)")

# Per-sample metrics
per_sample_records = []
for sid, dsub in df.groupby("sample"):
    if len(dsub) < min_spots_per_sample:
        continue
    if dsub[ct_nhood_key].nunique() < 2 or dsub[ecm_nhood_key].nunique() < 2:
        continue
    per_sample_records.append({
        "sample": sid,
        "n_spots": len(dsub),
        "nmi": normalized_mutual_info_score(dsub[ct_nhood_key], dsub[ecm_nhood_key]),
        "ari": adjusted_rand_score(dsub[ct_nhood_key], dsub[ecm_nhood_key]),
    })
per_sample = pd.DataFrame(per_sample_records)
print(f"\nPer-sample (n={len(per_sample)} samples):")
print(f"  NMI: median={per_sample['nmi'].median():.3f}, "
      f"IQR=[{per_sample['nmi'].quantile(0.25):.3f}, {per_sample['nmi'].quantile(0.75):.3f}]")
print(f"  ARI: median={per_sample['ari'].median():.3f}, "
      f"IQR=[{per_sample['ari'].quantile(0.25):.3f}, {per_sample['ari'].quantile(0.75):.3f}]")

# Joint frequency matrix (sample-averaged) in PERCENT units throughout
ct_labels  = sorted(df[ct_nhood_key].unique(),  key=lambda x: int(x))
ecm_labels = sorted(df[ecm_nhood_key].unique(), key=lambda x: int(x))

def joint_freq_pct(dsub, ct_labels, ecm_labels):
    cm = pd.crosstab(
        dsub[ct_nhood_key], dsub[ecm_nhood_key], dropna=False,
    ).reindex(index=ct_labels, columns=ecm_labels, fill_value=0).to_numpy(dtype=float)
    total = cm.sum()
    return (cm / total * 100.0) if total > 0 else cm

joint_per_sample = []
for sid, dsub in df.groupby("sample"):
    if len(dsub) < min_spots_per_sample:
        continue
    joint_per_sample.append(joint_freq_pct(dsub, ct_labels, ecm_labels))

joint_mean   = np.mean(joint_per_sample, axis=0)        # in percent, sums to 100
ct_marginal  = joint_mean.sum(axis=1)                   # CT neighborhood prevalence (%)
ecm_marginal = joint_mean.sum(axis=0)                   # ECM neighborhood prevalence (%)

# ----- Figure 1: Joint frequency matrix with marginals -----
heatmap_cmap = LinearSegmentedColormap.from_list(
    "ct_ecm", ["#ffffff", "#d4ecf2", "#7ec1cf", "#2c7a8a", "#0d3a4a"], N=256,
)
bar_color = "#7ec1cf"

fig = plt.figure(figsize=(11, 9))
gs = GridSpec(
    2, 3,
    width_ratios=[6, 1.4, 0.25],
    height_ratios=[1.0, 6],
    wspace=0.08, hspace=0.08,
    left=0.10, right=0.93, top=0.92, bottom=0.10,
)

ax_top  = fig.add_subplot(gs[0, 0])
ax_main = fig.add_subplot(gs[1, 0])
ax_right = fig.add_subplot(gs[1, 1])
ax_cbar  = fig.add_subplot(gs[1, 2])

# Main heatmap
im = ax_main.imshow(joint_mean, aspect="auto", cmap=heatmap_cmap,
                     vmin=0, vmax=joint_mean.max())
ax_main.set_xticks(range(len(ecm_labels)))
ax_main.set_xticklabels([f"E{l}" for l in ecm_labels], fontsize=10)
ax_main.set_yticks(range(len(ct_labels)))
ax_main.set_yticklabels([f"CT{l}" for l in ct_labels], fontsize=10)
ax_main.set_xlabel("ECM neighborhood", fontsize=11)
ax_main.set_ylabel("Cell-type neighborhood", fontsize=11)
ax_main.tick_params(length=0)
ax_main.grid(False)
for spine in ax_main.spines.values():
    spine.set_visible(False)

# Cell annotations: only label cells above a small threshold
threshold = max(joint_mean.max() * 0.05, 0.5)
for i in range(len(ct_labels)):
    for j in range(len(ecm_labels)):
        v = joint_mean[i, j]
        if v >= threshold:
            ax_main.text(j, i, f"{v:.1f}", ha="center", va="center",
                         fontsize=8,
                         color="white" if v > joint_mean.max() * 0.55 else "#222222")

# Top marginal: ECM neighborhood prevalence
ax_top.bar(range(len(ecm_labels)), ecm_marginal,
           color=bar_color, edgecolor="none", width=0.85)
ax_top.set_xlim(ax_main.get_xlim())
ax_top.set_xticks([])
ax_top.set_ylabel("ECM\n%", fontsize=9, rotation=0, ha="right", va="center", labelpad=12)
ax_top.tick_params(axis="y", labelsize=8)
ax_top.grid(False)
for s in ["top", "right", "bottom"]:
    ax_top.spines[s].set_visible(False)

# Right marginal: CT neighborhood prevalence
ax_right.barh(range(len(ct_labels)), ct_marginal,
              color=bar_color, edgecolor="none", height=0.85)
ax_right.set_ylim(ax_main.get_ylim())
ax_right.set_yticks([])
ax_right.set_xlabel("CT %", fontsize=9)
ax_right.tick_params(axis="x", labelsize=8)
ax_right.grid(False)
for s in ["top", "right", "left"]:
    ax_right.spines[s].set_visible(False)

# Colorbar
cbar = plt.colorbar(im, cax=ax_cbar)
cbar.set_label("Joint frequency (%)", fontsize=10)
cbar.ax.tick_params(labelsize=8)

fig.suptitle(
    f"Cell-type vs ECM neighborhood concordance — joint frequency\n"
    f"global NMI={nmi_global:.3f}, ARI={ari_global:.3f}, n={len(per_sample)} samples",
    fontsize=12, fontweight="semibold",
)
joint_df = pd.DataFrame(joint_mean, index=[f"CT{l}" for l in ct_labels], columns=[f"E{l}" for l in ecm_labels])
output.save(OUT, "celltype_ECM_neighborhood_joint_frequency", fig, joint_df, panel="Supp Fig 4g")
plt.show()

# ----- Figure 2: Per-sample NMI and ARI distributions -----
fig2, (ax_nmi, ax_ari) = plt.subplots(1, 2, figsize=(11, 4))

for ax, col, color, label in [
    (ax_nmi, "nmi", "#2c7a8a", "NMI"),
    (ax_ari, "ari", "#a85a3c", "ARI"),
]:
    vals = per_sample[col]
    ax.hist(vals, bins=25, color=color, edgecolor="white", linewidth=0.5, alpha=0.85)
    global_val = nmi_global if col == "nmi" else ari_global
    ax.axvline(global_val, color="#c0392b", linestyle="--", linewidth=1.5,
               label=f"Global = {global_val:.3f}")
    ax.axvline(vals.median(), color="#222222", linestyle=":", linewidth=1.5,
               label=f"Median = {vals.median():.3f}")
    ax.set_xlabel(f"Per-sample {label}", fontsize=10)
    ax.set_ylabel("Number of samples", fontsize=10)
    ax.legend(loc="upper right", fontsize=9, frameon=False)
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize=9)

fig2.suptitle("Per-sample concordance distribution",
              fontsize=12, fontweight="semibold", y=1.02)
plt.tight_layout()
output.save(OUT, "celltype_ECM_neighborhood_NMI_per_sample", fig2, per_sample.set_index("sample"), panel="Supp Fig 4g per sample")
plt.show()

# Store results
adata.uns["ct_ecm_concordance"] = {
    "nmi_global": nmi_global,
    "ari_global": ari_global,
    "per_sample": per_sample,
    "joint_mean_pct": pd.DataFrame(
        joint_mean,
        index=[f"CT{l}" for l in ct_labels],
        columns=[f"E{l}" for l in ecm_labels],
    ),
    "ct_marginal_pct":  pd.Series(ct_marginal,  index=[f"CT{l}" for l in ct_labels]),
    "ecm_marginal_pct": pd.Series(ecm_marginal, index=[f"E{l}"  for l in ecm_labels]),
}
print("\nResults stored in adata.uns['ct_ecm_concordance']")

### Supp Fig 4a: dotplot of cell composition per neighborhood

In [ ]:
# ### Cell-type marker dotplot — all cell types
#
# Marker score (composition × log2 FC⁺) per niche. Color = marker score.
# Size = specificity (fraction of cell type's total cohort abundance in
# this niche). Bright + large = strong niche-defining marker.


# User config
nhood_key        = "celltype_neighborhood"
c2l_prefix       = "meanscell_abundance_w_sf_"
curation_col     = "celltype_level4_curated"
EXCLUDE_PATTERNS = ["doublet", "other", "unknown"]

# Sanity checks
if nhood_key not in adata.obs.columns:
    raise ValueError(f"{nhood_key} not found in adata.obs")
if "celltype_annotation_key" not in adata.uns:
    raise ValueError("adata.uns['celltype_annotation_key'] not found.")

celltype_key = adata.uns["celltype_annotation_key"]

# cell2location abundances summed per curated cell type
counts_curated = neighborhoods.c2l_curated_counts(adata.obs, celltype_key, curation_col, EXCLUDE_PATTERNS, verbose=False)

# Spots with valid neighborhood and abundance
nh = adata.obs[nhood_key].astype("string")
valid_mask = nh.notna() & (counts_curated.sum(axis=1) > 0)
counts_curated = counts_curated.loc[valid_mask]
nh_valid = nh.loc[valid_mask]

# Core matrices
abund_per_nh = counts_curated.groupby(nh_valid, observed=True).sum()
scores       = neighborhoods.marker_scores(abund_per_nh)
composition, specificity, marker_score = scores["composition"], scores["specificity"], scores["marker_score"]

print(f"Cell types: {composition.shape[1]}, neighborhoods: {composition.shape[0]}")

# Niche order — cluster on composition
nhood_order = sorted(composition.index, key=lambda x: int(x))
if len(nhood_order) > 2:
    Z = linkage(np.sqrt(composition.loc[nhood_order].values),
                method="average", metric="euclidean")
    nhood_order = [nhood_order[i] for i in leaves_list(Z)]

# Cell type order — by best-niche assignment under marker score, then by score
nhood_index_map  = {nh: i for i, nh in enumerate(nhood_order)}
best_niche_per_ct = marker_score.loc[nhood_order].idxmax(axis=0)
best_score_per_ct = marker_score.loc[nhood_order].max(axis=0)
ct_table = pd.DataFrame({
    "best_niche_idx": best_niche_per_ct.map(nhood_index_map),
    "best_score":     best_score_per_ct,
})
ct_order = ct_table.sort_values(
    by=["best_niche_idx", "best_score"],
    ascending=[True, False],
).index.tolist()

# Reorder for plotting
color_M = marker_score.loc[nhood_order, ct_order].values
size_M  = specificity .loc[nhood_order, ct_order].values

# Dot sizing
size_scale = 220.0
spec_max   = specificity.values.max()
dot_areas  = size_M / max(spec_max, 1e-9) * size_scale
dot_areas  = np.maximum(dot_areas, 3)

# Layout
n_niches = composition.shape[0]
n_cts    = len(ct_order)
fig_w    = max(14.0, 0.22 * n_cts + 4.0)
fig_h    = max(3.5,  0.32 * n_niches + 2.5)
fig, ax  = plt.subplots(figsize=(fig_w, fig_h))

xx, yy = np.meshgrid(np.arange(n_cts), np.arange(n_niches))

# Faint grid through dot centers
for x in range(n_cts):
    ax.axvline(x, color="#ededed", linewidth=0.4, zorder=1)
for y in range(n_niches):
    ax.axhline(y, color="#ededed", linewidth=0.4, zorder=1)

sc = ax.scatter(
    xx.ravel(), yy.ravel(),
    s=dot_areas.ravel(),
    c=color_M.ravel(),
    cmap="RdPu", vmin=0, vmax=color_M.max(),
    edgecolors="#333333", linewidths=0.3,
    zorder=3,
)

ax.set_xticks(range(n_cts))
ax.set_xticklabels(ct_order, fontsize=9, rotation=45, ha="right",
                   rotation_mode="anchor")
ax.set_xlabel("Cell type", fontsize=11, labelpad=8)
ax.set_yticks(range(n_niches))
ax.set_yticklabels([f"N{nh_id}" for nh_id in nhood_order],
                   fontsize=10, fontweight="semibold")
ax.set_ylabel("Neighborhood", fontsize=10, labelpad=8)
ax.set_xlim(-0.5, n_cts - 0.5)
ax.set_ylim(-0.5, n_niches - 0.5)
ax.invert_yaxis()
ax.tick_params(length=0)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)

ax.set_title(
    "Cell-type markers per neighborhood (all cell types)",
    fontsize=12, fontweight="semibold", loc="left", pad=8,
)

cbar = plt.colorbar(sc, ax=ax, fraction=0.012, pad=0.01,
                    shrink=0.85, anchor=(0.0, 0.5))
cbar.set_label("Marker score\n(composition × log₂ FC⁺)", fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Specificity size legend
candidates = [0.05, 0.1, 0.2, 0.4, 0.7]
legend_vals = [v for v in candidates if v <= spec_max * 1.05]
if not legend_vals:
    legend_vals = [round(spec_max * f, 2) for f in [0.25, 0.5, 1.0]]
legend_handles = [
    plt.scatter([], [],
                s=v / max(spec_max, 1e-9) * size_scale,
                c="#7a7a7a", edgecolors="#333333", linewidths=0.3,
                label=f"{v:.2f}")
    for v in legend_vals
]
sleg = ax.legend(
    handles=legend_handles,
    loc="upper left", bbox_to_anchor=(1.06, 1.0),
    title="Specificity\n(fraction of cell type)",
    title_fontsize=8, fontsize=8,
    frameon=False, labelspacing=0.7, borderpad=0.3,
    handletextpad=0.6,
)
sleg.get_title().set_multialignment("left")
ax.add_artist(sleg)

plt.tight_layout()
output.save(OUT, "celltype_neighborhood_markers_all", fig, marker_score.loc[nhood_order, ct_order], panel="Supp Fig 4a")
plt.show()

adata.uns[f"{nhood_key}_marker_score"] = marker_score
print(f"Marker score table stored in adata.uns['{nhood_key}_marker_score']")

### Fig 4a: subset plot of cell composition

As saved, this cell leaves out neighborhoods 0 and 6 and shows curated level-4 names. The published
panel shows CT_0 (every neighborhood but CT_6), level-4 scANVI names and 36 cell types; no saved code
draws it. The closest reconstruction from the frozen files (cell2location means, scANVI names,
doublet/other/unknown removed, `dotplot_composite_score` on all neighborhoods but 6) gives the 36
published cell types plus Mural_LEPR and Plasma_nos, with a different row order (q05 abundances
match less well). The published panel was therefore drawn from a slightly different abundance
table or labelling (the annotation file was rewritten in May 2026 when k = 7 and 8 were tried).

In [ ]:
# ### Cell-type marker dotplot — top markers from selected neighborhoods
#
# Same marker score (composition × log2 FC⁺) as the all-cell-types version,
# but cell types are restricted to the union of top-N markers from a chosen
# subset of neighborhoods. Use this to filter out poorly-characterized niches
# (here: niches 0 and 6) from contributing cell types to the plot, while
# still showing all niche rows so the reader can see how the selected
# markers behave globally.


# User config
nhood_key        = "celltype_neighborhood"
c2l_prefix       = "meanscell_abundance_w_sf_"
curation_col     = "celltype_level4_curated"
EXCLUDE_PATTERNS = ["doublet", "other", "unknown"]

TOP_N_PER_NICHE   = 5                                # markers per included niche
INCLUDED_NICHES   = ["1", "2", "3", "4", "5", "7", "8", "9"]   # niches whose top-N contribute
SHOW_EXCLUDED_ROWS = False                            # if True, plot all niche rows;
                                                     # if False, only included niches appear

# Sanity checks
if nhood_key not in adata.obs.columns:
    raise ValueError(f"{nhood_key} not found in adata.obs")
if "celltype_annotation_key" not in adata.uns:
    raise ValueError("adata.uns['celltype_annotation_key'] not found.")

celltype_key = adata.uns["celltype_annotation_key"]

# cell2location abundances summed per curated cell type
counts_curated = neighborhoods.c2l_curated_counts(adata.obs, celltype_key, curation_col, EXCLUDE_PATTERNS, verbose=False)

# Spots with valid neighborhood and abundance
nh = adata.obs[nhood_key].astype("string")
valid_mask = nh.notna() & (counts_curated.sum(axis=1) > 0)
counts_curated = counts_curated.loc[valid_mask]
nh_valid = nh.loc[valid_mask]

# Core matrices
abund_per_nh = counts_curated.groupby(nh_valid, observed=True).sum()
scores       = neighborhoods.marker_scores(abund_per_nh)
composition, specificity, marker_score = scores["composition"], scores["specificity"], scores["marker_score"]

# Validate that included niches exist
missing = [n for n in INCLUDED_NICHES if n not in marker_score.index]
if missing:
    raise ValueError(f"INCLUDED_NICHES contains values not in data: {missing}")

# Pick top N markers per included niche, take the union
top_celltypes = set()
for nh_id in INCLUDED_NICHES:
    top_n = marker_score.loc[nh_id].sort_values(ascending=False).head(TOP_N_PER_NICHE)
    top_celltypes.update(top_n.index)
print(f"Cell types in plot: {len(top_celltypes)} "
      f"(top {TOP_N_PER_NICHE} from {len(INCLUDED_NICHES)} niches)")

# Niche row order — cluster on composition (subset to plot rows)
all_niches = sorted(composition.index, key=lambda x: int(x))
plot_niches = all_niches if SHOW_EXCLUDED_ROWS else INCLUDED_NICHES

# Use the full-niche clustering for row order so it's stable across runs
if len(all_niches) > 2:
    Z = linkage(np.sqrt(composition.loc[all_niches].values),
                method="average", metric="euclidean")
    full_order = [all_niches[i] for i in leaves_list(Z)]
else:
    full_order = all_niches
nhood_order = [n for n in full_order if n in plot_niches]

# Cell type order — by best-niche assignment, restricted to included niches
included_idx_map = {nh: i for i, nh in enumerate(INCLUDED_NICHES)}
ms_included = marker_score.loc[INCLUDED_NICHES, list(top_celltypes)]
best_niche_per_ct = ms_included.idxmax(axis=0)
best_score_per_ct = ms_included.max(axis=0)
ct_table = pd.DataFrame({
    "best_niche_idx": best_niche_per_ct.map(included_idx_map),
    "best_score":     best_score_per_ct,
})
ct_order = ct_table.sort_values(
    by=["best_niche_idx", "best_score"],
    ascending=[True, False],
).index.tolist()

# Reorder for plotting
color_M = marker_score.loc[nhood_order, ct_order].values
size_M  = specificity .loc[nhood_order, ct_order].values

# Dot sizing — anchor to specificity max over the cell types we're plotting
# Power transform with exponent > 1 expands the differential: small values
# get smaller, large values get larger. Combined with a higher size_scale
# and lower minimum floor, this gives a much more dramatic size range.
size_scale  = 400.0
size_power  = 1.6
size_floor  = 1.0
spec_max    = specificity.loc[nhood_order, ct_order].values.max()
dot_areas   = (size_M / max(spec_max, 1e-9)) ** size_power * size_scale
dot_areas   = np.maximum(dot_areas, size_floor)

# Layout
n_niches = len(nhood_order)
n_cts    = len(ct_order)
fig_w    = max(10.0, 0.30 * n_cts + 4.0)
fig_h    = max(3.0,  0.32 * n_niches + 2.5)
fig, ax  = plt.subplots(figsize=(fig_w, fig_h))

xx, yy = np.meshgrid(np.arange(n_cts), np.arange(n_niches))

# Faint grid through dot centers
for x in range(n_cts):
    ax.axvline(x, color="#ededed", linewidth=0.4, zorder=1)
for y in range(n_niches):
    ax.axhline(y, color="#ededed", linewidth=0.4, zorder=1)

# Dim rows that are excluded (visual cue that they didn't contribute markers)
if SHOW_EXCLUDED_ROWS:
    for i, nh_id in enumerate(nhood_order):
        if nh_id not in INCLUDED_NICHES:
            ax.axhspan(i - 0.5, i + 0.5, color="#fafafa", zorder=0)

sc = ax.scatter(
    xx.ravel(), yy.ravel(),
    s=dot_areas.ravel(),
    c=color_M.ravel(),
    cmap="RdPu", vmin=0, vmax=color_M.max(),
    edgecolors="#333333", linewidths=0.3,
    zorder=3,
)

ax.set_xticks(range(n_cts))
ax.set_xticklabels(ct_order, fontsize=9, rotation=45, ha="right",
                   rotation_mode="anchor")
ax.set_xlabel("Cell type", fontsize=11, labelpad=8)
ax.set_yticks(range(n_niches))
yticklabels = [
    f"N{nh_id}" + ("" if nh_id in INCLUDED_NICHES else " (excl.)")
    for nh_id in nhood_order
]
ax.set_yticklabels(yticklabels, fontsize=10, fontweight="semibold")
ax.set_ylabel("Neighborhood", fontsize=10, labelpad=8)
ax.set_xlim(-0.5, n_cts - 0.5)
ax.set_ylim(-0.5, n_niches - 0.5)
ax.invert_yaxis()
ax.tick_params(length=0)
ax.grid(False)
for s in ax.spines.values():
    s.set_visible(False)

title = (
    f"Cell-type markers per neighborhood "
    f"(top {TOP_N_PER_NICHE} from N{', N'.join(INCLUDED_NICHES)})"
)
ax.set_title(title, fontsize=12, fontweight="semibold", loc="left", pad=8)

cbar = plt.colorbar(sc, ax=ax, fraction=0.012, pad=0.01,
                    shrink=0.85, anchor=(0.0, 0.5))
cbar.set_label("Marker score\n(composition × log₂ FC⁺)", fontsize=9)
cbar.ax.tick_params(labelsize=8)

# Specificity size legend
candidates = [0.05, 0.1, 0.2, 0.4, 0.7]
legend_vals = [v for v in candidates if v <= spec_max * 1.05]
if not legend_vals:
    legend_vals = [round(spec_max * f, 2) for f in [0.25, 0.5, 1.0]]
legend_handles = [
    plt.scatter([], [],
                s=(v / max(spec_max, 1e-9)) ** size_power * size_scale,
                c="#7a7a7a", edgecolors="#333333", linewidths=0.3,
                label=f"{v:.2f}")
    for v in legend_vals
]
sleg = ax.legend(
    handles=legend_handles,
    loc="upper left", bbox_to_anchor=(1.06, 1.0),
    title="Specificity\n(fraction of cell type)",
    title_fontsize=8, fontsize=8,
    frameon=False, labelspacing=0.7, borderpad=0.3,
    handletextpad=0.6,
)
sleg.get_title().set_multialignment("left")
ax.add_artist(sleg)

plt.tight_layout()
output.save(OUT, "celltype_neighborhood_markers_topN", fig, marker_score.loc[nhood_order, ct_order], panel="Fig 4a")
plt.show()

### Supp Fig 4b: distribution of neighborhoods across cohort

In [ ]:
# ### Cell-type neighborhood distribution across patients


# User config
nhood_key            = "celltype_neighborhood"
sample_key           = "sample"
normalize            = True
min_spots_per_sample = 200
INCLUDED_NICHES      = None          # None = all niches; or a list like ["1","2","3","4","5","7","8","9"]
cluster_samples      = True
cluster_neighborhoods = True
cmap_name            = "tab20"
fig_width, fig_height = 18, 7
legend_ncol          = 2

# Sanity checks
for c in (sample_key, nhood_key):
    if c not in adata.obs.columns:
        raise KeyError(f"{c} not in adata.obs")

if not adata.obs[nhood_key].dtype.name == "category":
    adata.obs[nhood_key] = adata.obs[nhood_key].astype("category")

# Build sample × neighborhood count matrix
df    = adata.obs[[sample_key, nhood_key]].dropna()
pivot = df.groupby([sample_key, nhood_key], observed=True).size().unstack(fill_value=0)

# Drop low-coverage samples
n_before = len(pivot)
pivot = pivot.loc[pivot.sum(axis=1) >= min_spots_per_sample]
print(f"Samples: {len(pivot)} of {n_before} retained "
      f"(min {min_spots_per_sample} spots per sample)")
if pivot.empty:
    raise ValueError("No samples retained after filtering.")

# Optionally restrict to a subset of neighborhoods
if INCLUDED_NICHES is not None:
    pivot = pivot[[c for c in pivot.columns if str(c) in INCLUDED_NICHES]]
    print(f"Restricted to {len(INCLUDED_NICHES)} neighborhoods: {INCLUDED_NICHES}")

# Normalize within each sample so bars sum to 1 (proportion view)
if normalize:
    row_sums = pivot.sum(axis=1).replace(0, np.nan)
    pivot    = pivot.div(row_sums, axis=0).fillna(0)

# Cluster neighborhoods (cosine distance — appropriate for compositional data)
if cluster_neighborhoods and pivot.shape[1] > 2:
    Zc          = linkage(pivot.T.values.astype(np.float32),
                          method="average", metric="cosine")
    order_cols  = dendrogram(Zc, no_plot=True)["leaves"]
    pivot       = pivot.iloc[:, order_cols]
else:
    pivot = pivot.loc[:, pivot.sum(0).sort_values(ascending=False).index]

# Cluster samples (cosine distance on niche distribution)
if cluster_samples and pivot.shape[0] > 2:
    Zs          = linkage(pivot.values.astype(np.float32),
                          method="average", metric="cosine")
    order_rows  = dendrogram(Zs, no_plot=True)["leaves"]
    pivot       = pivot.iloc[order_rows]

# Use colors consistent with the spatial plots if already cached, else generate
cats_key   = f"{nhood_key}_categories"
colors_key = f"{nhood_key}_colors"

if cats_key in adata.uns and colors_key in adata.uns:
    cats_global   = list(adata.uns[cats_key])
    colors_global = list(adata.uns[colors_key])
    cm = dict(zip(cats_global, colors_global))
else:
    cats_global   = adata.obs[nhood_key].cat.categories.astype(str).tolist()
    cmap          = plt.get_cmap(cmap_name, len(cats_global))
    colors_global = [to_hex(cmap(i)) for i in range(len(cats_global))]
    adata.uns[cats_key]   = cats_global
    adata.uns[colors_key] = colors_global
    cm = dict(zip(cats_global, colors_global))

colors = [cm.get(str(col), "#999999") for col in pivot.columns]

# Plot
fig, ax = plt.subplots(figsize=(fig_width, fig_height))
x      = np.arange(pivot.shape[0])
bottom = np.zeros(pivot.shape[0], dtype=float)

for col, color in zip(pivot.columns, colors):
    vals = pivot[col].to_numpy()
    ax.bar(x, vals, bottom=bottom, width=1.0, color=color,
           edgecolor="none", label=f"N{col}")
    bottom += vals

ax.set_xlim(-0.5, len(x) - 0.5)
ax.set_ylim(0, 1.0 if normalize else None)
ax.set_ylabel("Proportion of spots" if normalize else "Spot count", fontsize=11)
ax.set_xlabel(f"Patients (n={len(x)}, hierarchically clustered)", fontsize=11)

# Sparse x-tick labels — for cohorts of 200+ samples, dense labels become unreadable
step = max(1, len(x) // 30)
ax.set_xticks(x[::step])
ax.set_xticklabels(pivot.index[::step], rotation=90, fontsize=7)
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(length=0)
ax.grid(False)
for spine_name in ["top", "right"]:
    ax.spines[spine_name].set_visible(False)

ax.set_title(
    "Cell-type neighborhood composition per patient",
    fontsize=12, fontweight="semibold", loc="left", pad=8,
)

ax.legend(
    loc="center left", bbox_to_anchor=(1.0, 0.5),
    frameon=False, ncol=legend_ncol, fontsize=9,
    title="Neighborhood", title_fontsize=10,
    handlelength=1.0, handleheight=1.0, labelspacing=0.4,
)

plt.tight_layout()
output.save(OUT, "celltype_neighborhood_per_patient", fig, pivot, panel="Supp Fig 4b")
plt.show()

# Compare with the matrix of the notebook's last run
pub = pd.read_csv(config.CT_ANNOTATIONS_CSV.parent / "celltype_neighborhoods_by_patient_stacked_matrix.csv", index_col=0)
pub.columns = pub.columns.astype(str)
new = pivot.rename(columns=str)
print(f"Samples: {len(new)} (last run {len(pub)}); same order: {list(new.index) == list(pub.index)}; "
      f"max |Δ| = {(new - pub.loc[new.index, new.columns]).abs().max().max():.2e}")

In [ ]:
output.versions(["anndata", "scanpy", "squidpy", "scikit-learn"])